# UFC Fight Predictor

Interactive predictions using the trained model from `fight_prediction_model.ipynb`.
Just **Run All** and use the widgets below to predict fights.

In [1]:
import pandas as pd
import numpy as np
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

DATA_DIR = Path("../data/2026-03-18/prepared")
MODEL_DIR = Path("../models") / DATA_DIR.parent.name

# Find available models
model_files = sorted(MODEL_DIR.glob("*.joblib"))
if not model_files:
    raise FileNotFoundError(f"No models found in {MODEL_DIR}. Run fight_prediction_model.ipynb first.")
MODEL_PATH = model_files[0]
model = joblib.load(MODEL_PATH)

# Load data
fighter_features = pd.read_csv(DATA_DIR / "fighter_features.csv")
feature_cols = pd.read_csv(DATA_DIR / "feature_cols.csv", header=None)[0].tolist()
train = pd.read_csv(DATA_DIR / "train.csv")

# Fighter name list for autocomplete
fighter_names = sorted(fighter_features["name"].dropna().tolist())

# Feature name lists
profile_feature_names = ["height_in", "weight_lb", "reach_in", "age",
                         "slpm", "str_acc_dec", "sapm", "str_def_dec",
                         "td_avg", "td_acc_dec", "td_def_dec", "sub_avg"]

hist_feature_names = ["hist_fights", "hist_wins", "hist_losses", "hist_win_rate",
                      "hist_strike_acc", "hist_td_acc", "hist_sub_avg",
                      "hist_ctrl_avg", "hist_finish_rate"]

diff_feature_names = profile_feature_names + hist_feature_names

# Categorical column names
stance_cats_a = [c for c in feature_cols if c.startswith("stance_a_")]
stance_cats_b = [c for c in feature_cols if c.startswith("stance_b_")]
wc_cats = [c for c in feature_cols if c.startswith("weight_class_")]

# Medians for imputation
phys_medians = train[[f"{c}_a" for c in ["height_in", "weight_lb", "reach_in", "age"]]].median()
hist_rate_medians = train[[f"{c}_a" for c in ["hist_win_rate", "hist_strike_acc", "hist_td_acc",
                                               "hist_sub_avg", "hist_ctrl_avg", "hist_finish_rate"]]].median()

print(f"Loaded model: {MODEL_PATH.name}")
if len(model_files) > 1:
    print(f"Available models: {', '.join(f.name for f in model_files)}")
print(f"{len(fighter_names)} fighters available.")

Loaded model: gradient_boosting.joblib
2651 fighters available.


In [2]:
def find_fighter(name):
    """Find a fighter by name (case-insensitive partial match)."""
    matches = fighter_features[fighter_features["name"].str.contains(name, case=False, na=False)]
    if len(matches) == 0:
        raise ValueError(f"No fighter found matching '{name}'")
    if len(matches) > 1:
        exact = matches[matches["name"].str.lower() == name.lower()]
        if len(exact) == 1:
            return exact.iloc[0]
        names = ", ".join(matches["name"].tolist()[:10])
        raise ValueError(f"Ambiguous name '{name}' — matches: {names}")
    return matches.iloc[0]


def build_feature_row(fighter_a, fighter_b):
    """Build a single feature row for a matchup between two fighters."""
    row = {}

    for suffix, fighter in [("a", fighter_a), ("b", fighter_b)]:
        for col in profile_feature_names:
            val = fighter.get(col, np.nan)
            row[f"{col}_{suffix}"] = val if pd.notna(val) else np.nan

        for col in hist_feature_names:
            val = fighter.get(col, np.nan)
            row[f"{col}_{suffix}"] = val if pd.notna(val) else np.nan

        stance = fighter.get("stance", "Unknown")
        if pd.isna(stance):
            stance = "Unknown"
        cats = stance_cats_a if suffix == "a" else stance_cats_b
        for cat_col in cats:
            cat_val = cat_col.split(f"stance_{suffix}_", 1)[1]
            row[cat_col] = 1 if stance == cat_val else 0

    for col in diff_feature_names:
        a_val = row.get(f"{col}_a", 0) or 0
        b_val = row.get(f"{col}_b", 0) or 0
        row[f"{col}_diff"] = a_val - b_val

    for cat_col in wc_cats:
        row[cat_col] = 0

    # Impute missing values
    for col in ["height_in", "weight_lb", "reach_in", "age"]:
        for suffix in ["a", "b"]:
            key = f"{col}_{suffix}"
            if pd.isna(row.get(key)):
                row[key] = phys_medians[f"{col}_a"]

    for col in ["hist_win_rate", "hist_strike_acc", "hist_td_acc",
                "hist_sub_avg", "hist_ctrl_avg", "hist_finish_rate"]:
        for suffix in ["a", "b"]:
            key = f"{col}_{suffix}"
            if pd.isna(row.get(key)):
                row[key] = hist_rate_medians[f"{col}_a"]

    for col in ["hist_fights", "hist_wins", "hist_losses"]:
        for suffix in ["a", "b"]:
            key = f"{col}_{suffix}"
            if pd.isna(row.get(key)):
                row[key] = 0

    for col in profile_feature_names:
        for suffix in ["a", "b"]:
            key = f"{col}_{suffix}"
            if pd.isna(row.get(key)):
                row[key] = 0

    # Recompute diffs after imputation
    for col in diff_feature_names:
        row[f"{col}_diff"] = row[f"{col}_a"] - row[f"{col}_b"]

    return pd.DataFrame([row])[feature_cols]


def predict_matchup(fighter_a_name, fighter_b_name):
    """Predict the outcome of a fight. Returns dict with winner, probs, names."""
    fighter_a = find_fighter(fighter_a_name)
    fighter_b = find_fighter(fighter_b_name)

    X = build_feature_row(fighter_a, fighter_b)
    prob_a = model.predict_proba(X)[0, 1]
    prob_b = 1 - prob_a

    winner = fighter_a["name"] if prob_a > 0.5 else fighter_b["name"]
    confidence = max(prob_a, prob_b)

    return {
        "fighter_a": fighter_a["name"],
        "fighter_b": fighter_b["name"],
        "prob_a": prob_a,
        "prob_b": prob_b,
        "winner": winner,
        "confidence": confidence,
    }

## Single Fight Prediction

In [8]:
fighter_a_input = widgets.Combobox(
    placeholder="Fighter A",
    options=fighter_names,
    ensure_option=False,
    layout=widgets.Layout(width="250px"),
)
fighter_b_input = widgets.Combobox(
    placeholder="Fighter B",
    options=fighter_names,
    ensure_option=False,
    layout=widgets.Layout(width="250px"),
)
vs_label = widgets.HTML(value="<b style='font-size:16px; margin:0 10px;'>vs</b>")
predict_btn = widgets.Button(description="Predict", button_style="primary")
single_output = widgets.Output()


def on_predict_click(_):
    single_output.clear_output()
    with single_output:
        a_name = fighter_a_input.value.strip()
        b_name = fighter_b_input.value.strip()
        if not a_name or not b_name:
            print("Enter both fighter names.")
            return
        try:
            result = predict_matchup(a_name, b_name)
        except ValueError as e:
            print(str(e))
            return

        pct_a = result["prob_a"] * 100
        pct_b = result["prob_b"] * 100
        color_a = "#4CAF50" if result["prob_a"] > 0.5 else "#999"
        color_b = "#4CAF50" if result["prob_b"] > 0.5 else "#999"

        html = f"""
        <div style='font-family:monospace; max-width:500px; margin-top:10px;'>
            <h3>{result['fighter_a']}  vs  {result['fighter_b']}</h3>
            <div style='margin:8px 0;'>
                <div style='display:flex; align-items:center; margin:4px 0;'>
                    <span style='width:180px;'>{result['fighter_a']}</span>
                    <div style='flex:1; background:#eee; border-radius:4px; height:24px;'>
                        <div style='width:{pct_a}%; background:{color_a}; height:100%; border-radius:4px;
                                    display:flex; align-items:center; justify-content:center;
                                    color:white; font-size:12px; min-width:40px;'>
                            {pct_a:.1f}%
                        </div>
                    </div>
                </div>
                <div style='display:flex; align-items:center; margin:4px 0;'>
                    <span style='width:180px;'>{result['fighter_b']}</span>
                    <div style='flex:1; background:#eee; border-radius:4px; height:24px;'>
                        <div style='width:{pct_b}%; background:{color_b}; height:100%; border-radius:4px;
                                    display:flex; align-items:center; justify-content:center;
                                    color:white; font-size:12px; min-width:40px;'>
                            {pct_b:.1f}%
                        </div>
                    </div>
                </div>
            </div>
            <p><b>Predicted winner: {result['winner']}</b> ({result['confidence']:.1%} confidence)</p>
        </div>
        """
        display(HTML(html))


predict_btn.on_click(on_predict_click)

display(widgets.VBox([
    widgets.HBox([fighter_a_input, vs_label, fighter_b_input, predict_btn]),
    single_output,
]))

## Predict a Full Card

Predict an entire fight card and display results in a styled table.

In [5]:
card = [
    ("Islam Makhachev", "Arman Tsarukyan"),
    ("Merab Dvalishvili", "Umar Nurmagomedov"),
    ("Ciryl Gane", "Alexander Volkov"),
    ("Beneil Dariush", "Renato Moicano"),
    ("Movsar Evloev", "Ilia Topuria"),
]

rows_html = ""
for a_name, b_name in card:
    result = predict_matchup(a_name, b_name)
    pct_a = result["prob_a"] * 100
    pct_b = result["prob_b"] * 100
    winner_style_a = "font-weight:bold; color:#4CAF50;" if result["prob_a"] > 0.5 else ""
    winner_style_b = "font-weight:bold; color:#4CAF50;" if result["prob_b"] > 0.5 else ""
    rows_html += f"""
    <tr>
        <td style="padding:8px; {winner_style_a}">{result["fighter_a"]}</td>
        <td style="padding:8px; text-align:center;">{pct_a:.1f}%</td>
        <td style="padding:8px; text-align:center;">{pct_b:.1f}%</td>
        <td style="padding:8px; {winner_style_b}">{result["fighter_b"]}</td>
        <td style="padding:8px; text-align:center;">{result["confidence"]:.1%}</td>
    </tr>"""

html = f"""
<table style="font-family:monospace; border-collapse:collapse; margin:10px 0;">
<thead>
    <tr style="border-bottom:2px solid #333;">
        <th style="padding:8px; text-align:left;">Fighter A</th>
        <th style="padding:8px;">Win %</th>
        <th style="padding:8px;">Win %</th>
        <th style="padding:8px; text-align:left;">Fighter B</th>
        <th style="padding:8px;">Confidence</th>
    </tr>
</thead>
<tbody>{rows_html}</tbody>
</table>
"""
display(HTML(html))


Fighter A,Win %,Win %,Fighter B,Confidence
Islam Makhachev,48.1%,51.9%,Arman Tsarukyan,51.9%
Merab Dvalishvili,17.9%,82.1%,Umar Nurmagomedov,82.1%
Ciryl Gane,46.2%,53.8%,Alexander Volkov,53.8%
Beneil Dariush,54.4%,45.6%,Renato Moicano,54.4%
Movsar Evloev,40.6%,59.4%,Ilia Topuria,59.4%


## Limitations

- **No fight-camp or injury data** — the model only sees career stats, not current form or training changes.
- **Historical bias** — predictions reflect patterns from past UFC events and may not capture stylistic evolution.
- **Missing data** — fighters with few UFC bouts have sparse stats; the model imputes medians for missing values.
- **No context features** — factors like short-notice replacements, weight misses, or altitude are not included.